#DEMANDAS TI

### CONFIGURAÇÃO E CARREGAMENTO DE DATASET

In [22]:
# ============================================================
# Importa as bibliotecas do Python
# ============================================================

# Importar o pandas
import pandas as pd
import numpy as np

# Importar o LabelEncoder da biblioteca scikit-learn
from sklearn.preprocessing import LabelEncoder

# Importar a função de divisão de treino e teste
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Modelos
from sklearn.ensemble import RandomForestClassifier
from xgboost          import XGBClassifier
from sklearn.metrics  import (accuracy_score, precision_score,
                               recall_score, f1_score,
                               classification_report,  confusion_matrix, roc_auc_score)

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')




In [23]:
# ============================================================
# Carregar o dataset a partir do arquivo Excel
# ============================================================

# url do arquivo no GitHub
url_dataset = "https://raw.githubusercontent.com/gilbertoag2007/machine-learning-demandas-ti/main/DEMANDAS_DOWNSTREAM_V4.xlsx"

# Cria um dataframe com o conteúdo do dataset
colunas_desejadas = ["ID_DEMANDA", "SISTEMA", "NOME_EQUIPE", "TIPO_DEMANDA", "NUM_DEMANDA", "ANO_DEMANDA", "SITUACAO_DEMANDA","DAT_INICIAL_CLASSIFICACAO", "DAT_PREVISTA_INI_DEMANDA","DAT_PREVISTA_INI_REQUISITOS", "DAT_REAL_INI_REQ", "DAT_PREVISTA_FIM_REQUISITOS", "DAT_REAL_FIM_REQ", "DIAS_ATRASO_REQUISITOS", "DAT_PREVISTA_INI_DESENV", "DAT_REAL_INI_DESENV", "DAT_PREVISTA_FIM_DESENV", "DAT_REAL_FIM_DESENV","DIAS_ATRASO_DESENVOLVIMENTO", "DAT_PREVISTA_FIM_DEMANDA", "CLASSIFICACAO"]
df_original = pd.read_excel(url_dataset, usecols=colunas_desejadas )

# Lista as 5 primeiras colunas do dataframe.

df_original.head()

,ID_DEMANDA,SISTEMA,NOME_EQUIPE,TIPO_DEMANDA,NUM_DEMANDA,ANO_DEMANDA,SITUACAO_DEMANDA,DAT_INICIAL_CLASSIFICACAO,DAT_PREVISTA_INI_DEMANDA,DAT_PREVISTA_INI_REQUISITOS,...,DAT_PREVISTA_FIM_REQUISITOS,DAT_REAL_FIM_REQ,DIAS_ATRASO_REQUISITOS,DAT_PREVISTA_INI_DESENV,DAT_REAL_INI_DESENV,DAT_PREVISTA_FIM_DESENV,DAT_REAL_FIM_DESENV,DIAS_ATRASO_DESENVOLVIMENTO,DAT_PREVISTA_FIM_DEMANDA,CLASSIFICACAO
0,22391,CSA,Corporativo,ORIENTAÇÃO,797,2026,FINALIZADA,2026-05-14 16:00:50,2026-05-15,NaT,...,NaT,NaT,NaN,2026-05-15,2026-05-15 11:08:57,2026-05-18,2026-05-15 12:02:28,0,2026-05-18,NO PRAZO
1,22381,DPP,Downstream,BUG IMPEDITIVO,787,2026,FINALIZADA,2026-05-13 19:00:43,2026-05-14,NaT,...,NaT,NaT,NaN,2026-05-14,2026-05-15 16:34:46,2026-05-15,2026-05-15 18:24:44,0,2026-05-15,NO PRAZO
2,22366,I-SIMP (DPP),Downstream,BUG NÃO IMPEDITIVO,772,2026,FINALIZADA,2026-05-11 12:00:35,2026-05-12,NaT,...,NaT,NaT,NaN,2026-05-12,2026-05-12 14:17:40,2026-05-18,2026-05-13 10:13:32,0,2026-05-18,NO PRAZO
3,22360,DPP,Downstream,BUG NÃO IMPEDITIVO,766,2026,FINALIZADA,2026-05-08 14:00:27,2026-05-11,NaT,...,NaT,NaT,NaN,2026-05-11,2026-05-11 09:55:40,2026-05-15,2026-05-15 12:19:49,0,2026-05-15,NO PRAZO
4,22327,SIGAF,Downstream,MELHORIA PEQUENA,733,2026,FINALIZADA,2026-05-05 15:01:16,2026-05-08,NaT,...,NaT,NaT,NaN,2026-05-08,2026-05-11 15:49:19,2026-05-14,2026-05-11 16:27:17,0,2026-05-14,NO PRAZO


##AJUSTES INICIAIS NO DATAFRAME

In [24]:
# ============================================================
# TÉCNICA: Label Encoding (Codificação de Rótulos)
# ============================================================

# Cria uma cópia do dataframe original para preservá-lo intacto
# Todas as alterações serão feitas apenas no df_ajustado
df_ajustado = df_original.copy()

# Cria uma nova coluna numérica baseada na coluna STATUS_FINAL
# map() substitui cada valor categórico pelo número correspondente
df_ajustado['CLASSIFICACAO_FINAL_NUM'] = df_ajustado['CLASSIFICACAO'].map({
    'ATRASO'         : 1,
    'NO PRAZO': 0
})

# Variavel Target
target = "CLASSIFICACAO_FINAL_NUM"



In [25]:
# ============================================================
# TÉCNICA: One-Hot Encoding
# ============================================================

# Aplicar One-Hot Encoding na coluna SISTEMA
# pd.get_dummies() cria uma coluna binária (0 ou 1) para cada sistema único
# dtype=int garante que os valores sejam inteiros ao invés de booleanos
# Aplicar nas colunas categóricas sem ordem natural
for coluna in ['SISTEMA', 'TIPO_DEMANDA']:
    dummies = pd.get_dummies(df_ajustado[coluna], prefix=coluna, dtype=int)
    df_ajustado = pd.concat([df_ajustado, dummies], axis=1)
    df_ajustado = df_ajustado.drop(columns=[coluna])


In [26]:

# ============================================================
# CONVERTER COLUNAS DE DATA PARA DATETIME
# Necessário para realizar operações matemáticas entre datas
# ============================================================
colunas_data = [
    'DAT_INICIAL_CLASSIFICACAO',
    'DAT_PREVISTA_INI_DEMANDA',
    'DAT_PREVISTA_INI_REQUISITOS',
    'DAT_REAL_INI_REQ',
    'DAT_PREVISTA_FIM_REQUISITOS',
    'DAT_REAL_FIM_REQ',
    'DAT_PREVISTA_INI_DESENV',
    'DAT_REAL_INI_DESENV',
    'DAT_PREVISTA_FIM_DESENV',
    'DAT_REAL_FIM_DESENV',
    'DAT_PREVISTA_FIM_DEMANDA'
]

for coluna in colunas_data:
    df_ajustado[coluna] = pd.to_datetime(
        df_ajustado[coluna], dayfirst=True, errors='coerce'
    )


In [27]:
# FEATURE ENGENIER

# Inclusão de feature para quantidade de dias de atraso no inicio do desenvolvimento.
df_ajustado['ATRASO_INICIO_DESENV']  = (df_ajustado['DAT_REAL_INI_DESENV']      - df_ajustado['DAT_PREVISTA_INI_DESENV']).dt.days

# Atribui -1 quando DIAS_ATRASO_REQUISITOS for nulo
df_ajustado['DIAS_ATRASO_REQUISITOS'] = df_ajustado['DIAS_ATRASO_REQUISITOS'].fillna(-1)

df_ajustado.head(50)

,ID_DEMANDA,NOME_EQUIPE,NUM_DEMANDA,ANO_DEMANDA,SITUACAO_DEMANDA,DAT_INICIAL_CLASSIFICACAO,DAT_PREVISTA_INI_DEMANDA,DAT_PREVISTA_INI_REQUISITOS,DAT_REAL_INI_REQ,DAT_PREVISTA_FIM_REQUISITOS,...,SISTEMA_SIGAF,SISTEMA_SIMP,SISTEMA_SRD - GLP,SISTEMA_SRD - PR,TIPO_DEMANDA_BUG IMPEDITIVO,TIPO_DEMANDA_BUG NÃO IMPEDITIVO,TIPO_DEMANDA_MELHORIA MÉDIA,TIPO_DEMANDA_MELHORIA PEQUENA,TIPO_DEMANDA_ORIENTAÇÃO,ATRASO_INICIO_DESENV
0,22391,Corporativo,797,2026,FINALIZADA,2026-05-14 16:00:50,2026-05-15,NaT,NaT,NaT,...,0,0,0,0,0,0,0,0,1,0
1,22381,Downstream,787,2026,FINALIZADA,2026-05-13 19:00:43,2026-05-14,NaT,NaT,NaT,...,0,0,0,0,1,0,0,0,0,1
2,22366,Downstream,772,2026,FINALIZADA,2026-05-11 12:00:35,2026-05-12,NaT,NaT,NaT,...,0,0,0,0,0,1,0,0,0,0
3,22360,Downstream,766,2026,FINALIZADA,2026-05-08 14:00:27,2026-05-11,NaT,NaT,NaT,...,0,0,0,0,0,1,0,0,0,0
4,22327,Downstream,733,2026,FINALIZADA,2026-05-05 15:01:16,2026-05-08,NaT,NaT,NaT,...,1,0,0,0,0,0,0,1,0,3
5,22320,Downstream,726,2026,FINALIZADA,2026-05-04 14:11:25,2026-05-07,NaT,NaT,NaT,...,0,0,0,0,0,0,0,1,0,-3
6,22312,Downstream,718,2026,FINALIZADA,2026-04-30 12:01:00,2026-05-04,NaT,NaT,NaT,...,0,0,0,0,1,0,0,0,0,1
7,22307,Downstream,713,2026,FINALIZADA,2026-04-29 20:00:12,2026-04-30,NaT,NaT,NaT,...,0,0,0,0,0,0,0,0,1,0
8,22302,Corporativo,708,2026,FINALIZADA,2026-04-29 15:00:57,2026-04-30,NaT,NaT,NaT,...,0,0,0,0,0,1,0,0,0,4
9,22298,Corporativo,704,2026,FINALIZADA,2026-04-29 12:00:57,2026-04-30,NaT,NaT,NaT,...,0,0,0,0,0,1,0,0,0,0


In [28]:
# Lista colunas do dataset
print(df_ajustado.columns.tolist())

['ID_DEMANDA', 'NOME_EQUIPE', 'NUM_DEMANDA', 'ANO_DEMANDA', 'SITUACAO_DEMANDA', 'DAT_INICIAL_CLASSIFICACAO', 'DAT_PREVISTA_INI_DEMANDA', 'DAT_PREVISTA_INI_REQUISITOS', 'DAT_REAL_INI_REQ', 'DAT_PREVISTA_FIM_REQUISITOS', 'DAT_REAL_FIM_REQ', 'DIAS_ATRASO_REQUISITOS', 'DAT_PREVISTA_INI_DESENV', 'DAT_REAL_INI_DESENV', 'DAT_PREVISTA_FIM_DESENV', 'DAT_REAL_FIM_DESENV', 'DIAS_ATRASO_DESENVOLVIMENTO', 'DAT_PREVISTA_FIM_DEMANDA', 'CLASSIFICACAO', 'CLASSIFICACAO_FINAL_NUM', 'SISTEMA_CSA', 'SISTEMA_DFe - Documento de Fiscalização Eletrônico', 'SISTEMA_DPP ', 'SISTEMA_I-SIMP (DPP)', 'SISTEMA_RENOVACALC', 'SISTEMA_SIGAF', 'SISTEMA_SIMP', 'SISTEMA_SRD - GLP', 'SISTEMA_SRD - PR', 'TIPO_DEMANDA_BUG IMPEDITIVO', 'TIPO_DEMANDA_BUG NÃO IMPEDITIVO', 'TIPO_DEMANDA_MELHORIA MÉDIA', 'TIPO_DEMANDA_MELHORIA PEQUENA', 'TIPO_DEMANDA_ORIENTAÇÃO', 'ATRASO_INICIO_DESENV']


In [29]:
# Exclui as colunas não necessárias para predição.

# Colunas One-Hot Encoding geradas para SISTEMA e TIPO_DEMANDA
colunas_sistema = [col for col in df_ajustado.columns if col.startswith('SISTEMA_')]
colunas_tipo    = [col for col in df_ajustado.columns if col.startswith('TIPO_DEMANDA_')]

# Features numéricas
colunas_numericas = [
    'ATRASO_INICIO_DESENV',
    'DIAS_ATRASO_REQUISITOS'
]

# Concatena todas as colunas necessárias
colunas_modelo = colunas_sistema + colunas_tipo + colunas_numericas + [target]

# Filtra o dataset
df_ajustado = df_ajustado[colunas_modelo]

# Separa features e target
X = df_ajustado.drop(columns=[target])
y = df_ajustado[target]

print(f'Features: {X.shape[1]} colunas')
print(f'Registros: {X.shape[0]}')
print(f'\nColunas utilizadas:\n{X.columns.tolist()}')
print(f'\nDistribuição do target:\n{y.value_counts()}')

Features: 16 colunas
Registros: 1632

Colunas utilizadas:
['SISTEMA_CSA', 'SISTEMA_DFe - Documento de Fiscalização Eletrônico', 'SISTEMA_DPP ', 'SISTEMA_I-SIMP (DPP)', 'SISTEMA_RENOVACALC', 'SISTEMA_SIGAF', 'SISTEMA_SIMP', 'SISTEMA_SRD - GLP', 'SISTEMA_SRD - PR', 'TIPO_DEMANDA_BUG IMPEDITIVO', 'TIPO_DEMANDA_BUG NÃO IMPEDITIVO', 'TIPO_DEMANDA_MELHORIA MÉDIA', 'TIPO_DEMANDA_MELHORIA PEQUENA', 'TIPO_DEMANDA_ORIENTAÇÃO', 'ATRASO_INICIO_DESENV', 'DIAS_ATRASO_REQUISITOS']

Distribuição do target:
CLASSIFICACAO_FINAL_NUM
0    1470
1     162
Name: count, dtype: int64


In [30]:
# Exibir resumo das colunas geradas e seus tipos
print('📊 Colunas do dataframe ajustado:')
print(df_ajustado.dtypes)
print(f'\n✅ Shape final: {df_ajustado.shape[0]} linhas x {df_ajustado.shape[1]} colunas')


📊 Colunas do dataframe ajustado:
SISTEMA_CSA                                             int64
SISTEMA_DFe - Documento de Fiscalização Eletrônico      int64
SISTEMA_DPP                                             int64
SISTEMA_I-SIMP (DPP)                                    int64
SISTEMA_RENOVACALC                                      int64
SISTEMA_SIGAF                                           int64
SISTEMA_SIMP                                            int64
SISTEMA_SRD - GLP                                       int64
SISTEMA_SRD - PR                                        int64
TIPO_DEMANDA_BUG IMPEDITIVO                             int64
TIPO_DEMANDA_BUG NÃO IMPEDITIVO                         int64
TIPO_DEMANDA_MELHORIA MÉDIA                             int64
TIPO_DEMANDA_MELHORIA PEQUENA                           int64
TIPO_DEMANDA_ORIENTAÇÃO                                 int64
ATRASO_INICIO_DESENV                                    int64
DIAS_ATRASO_REQUISITOS               

#ANALISE DOS DADOS

In [31]:
# ============================================================
# VERIFICAR O BALANCEAMENTO DA COLUNA TARGET
# ============================================================

balanceamento = df_ajustado[target].value_counts()
percentual    = df_ajustado[target].value_counts(normalize=True) * 100

# Exibir resultado
print('Distribuição da variável target:\n')
print(f'🟢 DENTRO DO PRAZO (0): {balanceamento[0]} registros ({percentual[0]:.1f}%)')
print(f'🔴 ATRASO          (1): {balanceamento[1]} registros ({percentual[1]:.1f}%)')

Distribuição da variável target:

🟢 DENTRO DO PRAZO (0): 1470 registros (90.1%)
🔴 ATRASO          (1): 162 registros (9.9%)


#TESTANDO OS MODELOS

In [32]:


# ----------------------------------------------------------
# Dividir em treino (80%) e teste (20%)
# stratify=y garante que a proporção de 0 e 1 seja mantida
# igual nos dois conjuntos — essencial para dados desbalanceados
# random_state=42 garante que a divisão seja reproduzível
# ----------------------------------------------------------
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size    = 0.20,
    stratify     = y,
    random_state = 42
)

# Exibir o resultado da divisão
print(f'Total de registros  : {len(X)}')
print(f'Registros de treino : {len(X_treino)} ({len(X_treino)/len(X)*100:.1f}%)')
print(f'Registros de teste  : {len(X_teste)} ({len(X_teste)/len(X)*100:.1f}%)')
print(f'\nDistribuição do target no treino:\n{y_treino.value_counts()}')
print(f'\nDistribuição do target no teste:\n{y_teste.value_counts()}')

Total de registros  : 1632
Registros de treino : 1305 (80.0%)
Registros de teste  : 327 (20.0%)

Distribuição do target no treino:
CLASSIFICACAO_FINAL_NUM
0    1175
1     130
Name: count, dtype: int64

Distribuição do target no teste:
CLASSIFICACAO_FINAL_NUM
0    295
1     32
Name: count, dtype: int64


In [33]:
# ─────────────────────────────────────────────
# 2. SMOTE — cria amostras sintéticas da classe minoritária
#    Só aplicar no conjunto de TREINO, nunca no teste!
# ─────────────────────────────────────────────
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_treino, y_treino)

print(f"Antes do SMOTE : {y_treino.value_counts().to_dict()}")
print(f"Depois do SMOTE: {y_train_bal.value_counts().to_dict()}")

Antes do SMOTE : {0: 1175, 1: 130}
Depois do SMOTE: {0: 1175, 1: 1175}


In [34]:
# ── 2. Divisão treino e teste ─────────────────────────────────────────────────
# stratify=y garante proporção 80/20 em ambos os conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Treino : {X_train.shape[0]} registros')
print(f'Teste  : {X_test.shape[0]} registros')
print(f'\nDistribuição no treino:\n{y_train.value_counts()}')
print(f'\nDistribuição no teste:\n{y_test.value_counts()}')

Treino : 1305 registros
Teste  : 327 registros

Distribuição no treino:
CLASSIFICACAO_FINAL_NUM
0    1175
1     130
Name: count, dtype: int64

Distribuição no teste:
CLASSIFICACAO_FINAL_NUM
0    295
1     32
Name: count, dtype: int64


In [35]:
# ── 3. Definição dos modelos e variações de balanceamento ─────────────────────
modelos = {

    # Baseline simples — sem balanceamento
    'Regressão Logística (sem balanceamento)': (
        LogisticRegression(max_iter=1000, random_state=42),
        'sem_smote', None
    ),

    # Regressão Logística com penalização da classe minoritária
    'Regressão Logística (class_weight)': (
        LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'sem_smote', None
    ),

    # Random Forest sem balanceamento — baseline do modelo principal
    'Random Forest (sem balanceamento)': (
        RandomForestClassifier(n_estimators=200, random_state=42),
        'sem_smote', None
    ),

    # Random Forest com penalização — alternativa simples ao SMOTE
    'Random Forest (class_weight)': (
        RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
        'sem_smote', None
    ),

    # Random Forest com SMOTE — dados sintéticos para balancear
    'Random Forest (SMOTE)': (
        RandomForestClassifier(n_estimators=200, random_state=42),
        'smote', None
    ),

    # XGBoost sem balanceamento
    'XGBoost (sem balanceamento)': (
        XGBClassifier(n_estimators=200, eval_metric='logloss', random_state=42),
        'sem_smote', None
    ),

    # XGBoost com scale_pos_weight — equivalente ao class_weight para XGBoost
    # scale_pos_weight = qtd negativos / qtd positivos
    'XGBoost (scale_pos_weight)': (
        XGBClassifier(
            n_estimators=200,
            scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
            eval_metric='logloss',
            random_state=42
        ),
        'sem_smote', None
    ),

    # XGBoost com SMOTE
    'XGBoost (SMOTE)': (
        XGBClassifier(n_estimators=200, eval_metric='logloss', random_state=42),
        'smote', None
    ),
}

In [36]:
# ── 4. Treinamento e avaliação ─────────────────────────────────────────────────
resultados = []

for nome, (modelo, estrategia, _) in modelos.items():

    # Aplica SMOTE apenas no treino quando indicado
    if estrategia == 'smote':
        X_treino_bal, y_treino_bal = SMOTE(random_state=42).fit_resample(X_train, y_train)
    else:
        X_treino_bal, y_treino_bal = X_train, y_train

    # Treina o modelo
    modelo.fit(X_treino_bal, y_treino_bal)

    # Predições no conjunto de teste (dados reais, nunca balanceados)
    y_pred      = modelo.predict(X_test)
    y_pred_prob = modelo.predict_proba(X_test)[:, 1]

    # Métricas focadas na classe minoritária ATRASO
    report = classification_report(y_test, y_pred, output_dict=True)
    auc    = roc_auc_score(y_test, y_pred_prob)

    # Identifica qual label corresponde a ATRASO (1 ou 0 dependendo do LabelEncoder)
    classe_atraso = str(y_test.unique().max())  # assume que ATRASO = valor maior

    resultados.append({
        'Modelo'              : nome,
        'F1 ATRASO'           : round(report[classe_atraso]['f1-score'], 3),
        'Precision ATRASO'    : round(report[classe_atraso]['precision'], 3),
        'Recall ATRASO'       : round(report[classe_atraso]['recall'], 3),
        'AUC-ROC'             : round(auc, 3),
        'Acurácia'            : round(report['accuracy'], 3),
    })

    print(f'\n{"="*60}')
    print(f'Modelo: {nome}')
    print(f'{"="*60}')
    print(classification_report(y_test, y_pred, target_names=['NO PRAZO','ATRASO']))
    print(f'AUC-ROC: {auc:.3f}')
    print(f'Matriz de Confusão:\n{confusion_matrix(y_test, y_pred)}')



Modelo: Regressão Logística (sem balanceamento)
              precision    recall  f1-score   support

    NO PRAZO       0.92      1.00      0.96       295
      ATRASO       0.89      0.25      0.39        32

    accuracy                           0.92       327
   macro avg       0.91      0.62      0.67       327
weighted avg       0.92      0.92      0.90       327

AUC-ROC: 0.879
Matriz de Confusão:
[[294   1]
 [ 24   8]]

Modelo: Regressão Logística (class_weight)
              precision    recall  f1-score   support

    NO PRAZO       0.97      0.87      0.92       295
      ATRASO       0.38      0.75      0.51        32

    accuracy                           0.86       327
   macro avg       0.68      0.81      0.71       327
weighted avg       0.91      0.86      0.88       327

AUC-ROC: 0.882
Matriz de Confusão:
[[256  39]
 [  8  24]]

Modelo: Random Forest (sem balanceamento)
              precision    recall  f1-score   support

    NO PRAZO       0.95      1.00      

In [37]:
# ── 5. Tabela comparativa dos resultados ──────────────────────────────────────
# Ordenada pelo F1-Score da classe ATRASO — métrica mais importante
df_resultados = pd.DataFrame(resultados).sort_values('F1 ATRASO', ascending=False)

print('\n\n══════════════════════════════════════════════════════════════')
print('COMPARATIVO GERAL — ordenado por F1 ATRASO')
print('══════════════════════════════════════════════════════════════')
print(df_resultados.to_string(index=False))



══════════════════════════════════════════════════════════════
COMPARATIVO GERAL — ordenado por F1 ATRASO
══════════════════════════════════════════════════════════════
                                 Modelo  F1 ATRASO  Precision ATRASO  Recall ATRASO  AUC-ROC  Acurácia
      Random Forest (sem balanceamento)      0.625             0.938          0.469    0.821     0.945
            XGBoost (sem balanceamento)      0.612             0.882          0.469    0.811     0.942
                  Random Forest (SMOTE)      0.590             0.621          0.562    0.810     0.924
                        XGBoost (SMOTE)      0.576             0.630          0.531    0.784     0.924
     Regressão Logística (class_weight)      0.505             0.381          0.750    0.882     0.856
             XGBoost (scale_pos_weight)      0.466             0.415          0.531    0.797     0.881
           Random Forest (class_weight)      0.441             0.417          0.469    0.804     0.884
Regre

In [39]:
# Ajustar de Threshold

modelo_rf = RandomForestClassifier(n_estimators=200, random_state=42)
modelo_rf.fit(X_train, y_train)
y_prob = modelo_rf.predict_proba(X_test)[:, 1]

print(f'{"Threshold":<12} {"F1":<8} {"Precision":<12} {"Recall":<10} {"AUC-ROC"}')
print('-' * 55)

for threshold in [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]:
    y_pred_t = (y_prob >= threshold).astype(int)
    report   = classification_report(y_test, y_pred_t, output_dict=True)
    classe   = str(y_test.unique().max())
    print(f'{threshold:<12} '
          f'{report[classe]["f1-score"]:<8.3f} '
          f'{report[classe]["precision"]:<12.3f} '
          f'{report[classe]["recall"]:<10.3f} '
          f'{roc_auc_score(y_test, y_prob):.3f}')

Threshold    F1       Precision    Recall     AUC-ROC
-------------------------------------------------------
0.5          0.625    0.938        0.469      0.821
0.45         0.612    0.882        0.469      0.821
0.4          0.600    0.833        0.469      0.821
0.35         0.577    0.750        0.469      0.821
0.3          0.566    0.714        0.469      0.821
0.25         0.586    0.654        0.531      0.821
0.2          0.610    0.667        0.562      0.821


In [40]:
for threshold in [0.20, 0.15, 0.10, 0.08, 0.05]:
    y_pred_t = (y_prob >= threshold).astype(int)
    report   = classification_report(y_test, y_pred_t, output_dict=True)
    classe   = str(y_test.unique().max())
    print(f'Threshold {threshold} → '
          f'Precision={report[classe]["precision"]:.3f} | '
          f'Recall={report[classe]["recall"]:.3f} | '
          f'F1={report[classe]["f1-score"]:.3f}')

Threshold 0.2 → Precision=0.667 | Recall=0.562 | F1=0.610
Threshold 0.15 → Precision=0.413 | Recall=0.594 | F1=0.487
Threshold 0.1 → Precision=0.400 | Recall=0.625 | F1=0.488
Threshold 0.08 → Precision=0.370 | Recall=0.625 | F1=0.465
Threshold 0.05 → Precision=0.278 | Recall=0.688 | F1=0.396
